# Figure 2(b)-Style Generalizability With A PCA Encoder

This notebook is the PCA-encoder version of `generalizability_figure.ipynb`.

The SSCD notebook asks: does a generated image look like a near-copy of any training image in SSCD feature space? This notebook asks the same question in a **domain-local PCA feature space** fitted on normalized CAMELS training slices.

For each generated sample `x`, compute its maximum cosine similarity to the real training samples in PCA space. The copy threshold is calibrated from real-vs-real nearest-neighbor similarities:

```text
threshold = quantile(max_j!=i cosine(PCA(real_i), PCA(real_j)), PCA_COPY_QUANTILE)
PCA_GL = 1 - fraction(max_i cosine(PCA(generated), PCA(real_i)) >= threshold)
```

This is not a replacement for SSCD or P(k). It is a useful sanity-check encoder because it uses the same normalized fields as training instead of natural-image features.

## Configuration

Default is U64-only, matching the runs we were actively checking. Set `GENERALIZABILITY_ARCHES="u64,u128,u256"` before launching the notebook if those sample files exist.

Important defaults:

- `PCA_FIT_MODE = "largest"`: fit one PCA basis per architecture on the largest available real training set, then reuse that fixed basis across dataset sizes. This makes the x-axis comparison less arbitrary than fitting a separate PCA basis for every run.
- `PCA_COPY_QUANTILE = 0.99`: generated samples above the 99th percentile of real-real nearest-neighbor similarity are counted copy-like.
- `PCA_MAX_REAL_COMPARE = 4096`: caps the real training slices used in the nearest-neighbor comparison so full-size runs do not build huge similarity matrices. Increase this for final production if memory allows.

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

# Robust project-root detection for Great Lakes and local copies of the notebook.
PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/home/jiamingp/Diffusion_model'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'simdiff_eval').exists()), PROJECT_CANDIDATES[0])

for candidate in (PROJECT_DIR, PROJECT_DIR / 'cosmo_diffusion'):
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from simdiff_eval.io import as_nchw, load_npy, load_real_from_config

ARCHES = os.environ.get('GENERALIZABILITY_ARCHES', 'u64').split(',')
ARCHES = [arch.strip() for arch in ARCHES if arch.strip()]
ARCH_LABELS = {'u64': 'UNet-64', 'u128': 'UNet-128', 'u256': 'UNet-256'}
ARCH_COLORS = {'u64': 'tab:red', 'u128': 'tab:blue', 'u256': 'limegreen'}
ARCH_MARKERS = {'u64': 'o', 'u128': 's', 'u256': '^'}

SEED = int(os.environ.get('GENERALIZABILITY_SEED', 123))
PCA_N_COMPONENTS = int(os.environ.get('PCA_N_COMPONENTS', 32))
PCA_MAX_FIT_REAL = int(os.environ.get('PCA_MAX_FIT_REAL', 1024))
PCA_MAX_REAL_COMPARE = int(os.environ.get('PCA_MAX_REAL_COMPARE', 4096))
PCA_MAX_GENERATED = None if os.environ.get('PCA_MAX_GENERATED', '').strip() == '' else int(os.environ['PCA_MAX_GENERATED'])
PCA_COPY_QUANTILE = float(os.environ.get('PCA_COPY_QUANTILE', 0.99))
PCA_FIT_MODE = os.environ.get('PCA_FIT_MODE', 'largest')  # 'largest' or 'per_run'
SIMILARITY_BATCH_SIZE = int(os.environ.get('PCA_SIMILARITY_BATCH_SIZE', 1024))

MANIFEST_CANDIDATES = [
    PROJECT_DIR / 'local' / 'fig1_lh' / 'manifest.json',
    PROJECT_DIR / 'configs' / 'templates' / 'reproducibility_manifest_template.json',
]
MANIFEST_PATH = next((p for p in MANIFEST_CANDIDATES if p.exists()), MANIFEST_CANDIDATES[0])
CONFIG_DIR = PROJECT_DIR / 'local' / 'fig1_lh' / 'configs'
SAMPLE_ROOTS = [
    PROJECT_DIR / 'results' / 'fig1_lh' / 'samples',
    PROJECT_DIR / 'results' / 'tables' / 'samples',
    PROJECT_DIR / 'results' / 'samples',
]
OUTPUT_DIR = PROJECT_DIR / 'results' / 'figures'
TABLE_DIR = PROJECT_DIR / 'results' / 'tables'

print('project:', PROJECT_DIR)
print('manifest:', MANIFEST_PATH)
print('arches:', ARCHES)
print('PCA components:', PCA_N_COMPONENTS)
print('PCA fit mode:', PCA_FIT_MODE)
print('PCA fit slice cap:', PCA_MAX_FIT_REAL)
print('real comparison slice cap:', PCA_MAX_REAL_COMPARE)
print('generated cap:', PCA_MAX_GENERATED)
print('copy threshold quantile:', PCA_COPY_QUANTILE)
print('sample roots:')
for root in SAMPLE_ROOTS:
    print(' ', root, 'exists=', root.exists())

## Discover Runs

The table below checks config and generated-sample availability. Missing sample files are skipped in the metric cell.

In [ ]:
def dataset_size(row: dict[str, Any]) -> float:
    for key in ('dataset_size', 'actual_2d', 'target_2d'):
        value = row.get(key)
        if value is not None:
            return float(value)
    raise ValueError(f"No dataset-size field in row: {row}")


def config_path_for(row: dict[str, Any]) -> Path:
    if row.get('config'):
        path = Path(row['config'])
        if not path.is_absolute():
            path = PROJECT_DIR / path
        return path
    return CONFIG_DIR / f"{row['run_name']}.yaml"


def sample_path_for(row: dict[str, Any]) -> Path | None:
    if row.get('sample_path'):
        raw = str(row['sample_path']).format(seed=SEED, run_name=row['run_name'])
        path = Path(raw)
        if not path.is_absolute():
            path = PROJECT_DIR / path
        if path.exists():
            return path
    for root in SAMPLE_ROOTS:
        for suffix in ('.npy', '.npz'):
            path = root / f"{row['run_name']}_seed{SEED}{suffix}"
            if path.exists():
                return path
    return None


def raw_sim_cap_for(row: dict[str, Any], slice_cap: int | None) -> int | None:
    if slice_cap is None:
        return None
    slices_per_sim = row.get('slices_per_sim')
    if slices_per_sim is None:
        zthin = int(row.get('zthin', 4) or 4)
        slices_per_sim = max(1, 128 // zthin)
    cap = max(1, int(math.ceil(int(slice_cap) / int(slices_per_sim))))
    total_raw = row.get('n_samples_simulations')
    if total_raw is not None:
        cap = min(cap, int(total_raw))
    return cap


def load_sample_array(path: Path) -> np.ndarray:
    if path.suffix == '.npz':
        z = np.load(path, mmap_mode='r')
        try:
            if 'samples' in z:
                return np.asarray(z['samples'])
            if 'arr_0' in z:
                return np.asarray(z['arr_0'])
            first_key = list(z.files)[0]
            return np.asarray(z[first_key])
        finally:
            z.close()
    return load_npy(path)


manifest = json.loads(MANIFEST_PATH.read_text())
rows = sorted([row for row in manifest if row.get('arch') in ARCHES], key=lambda row: (row.get('arch', ''), dataset_size(row)))
if not rows:
    raise RuntimeError(f'No manifest rows found for ARCHES={ARCHES!r}')

status_rows = []
for row in rows:
    config_path = config_path_for(row)
    sample_path = sample_path_for(row)
    status_rows.append({
        'arch': row.get('arch'),
        'run_name': row['run_name'],
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'real_compare_raw_sim_cap': raw_sim_cap_for(row, PCA_MAX_REAL_COMPARE),
        'pca_fit_raw_sim_cap': raw_sim_cap_for(row, PCA_MAX_FIT_REAL),
        'config_exists': config_path.exists(),
        'sample_exists': sample_path is not None,
        'config_path': str(config_path),
        'sample_path': str(sample_path) if sample_path is not None else None,
    })

status_df = pd.DataFrame(status_rows).sort_values(['arch', 'dataset_size']).reset_index(drop=True)
display(status_df)
print('available sample rows:', int(status_df['sample_exists'].sum()), '/', len(status_df))

## PCA Encoder And Metric

The PCA encoder operates on flattened normalized fields `(C, H, W)`. Embeddings are L2-normalized before cosine similarity. The copy threshold is not a fixed number like SSCD's 0.6; it is calibrated from each run's real training set in the same PCA space.

In [ ]:
def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return arr[idx].copy()


def flatten_images(images: np.ndarray) -> np.ndarray:
    arr = as_nchw(images).astype(np.float32, copy=False)
    return arr.reshape(len(arr), -1)


class PCAEncoder:
    def __init__(self, mean: np.ndarray, scale: np.ndarray, components: np.ndarray, explained_variance_ratio: np.ndarray):
        self.mean = mean.astype(np.float32)
        self.scale = scale.astype(np.float32)
        self.components = components.astype(np.float32)
        self.explained_variance_ratio = explained_variance_ratio.astype(np.float32)

    def transform(self, images: np.ndarray) -> np.ndarray:
        x = flatten_images(images)
        x = (x - self.mean) / self.scale
        return x @ self.components.T


def fit_pca_encoder(real: np.ndarray, n_components: int = 32, max_fit: int = 1024) -> PCAEncoder:
    x = flatten_images(evenly_limit(real, max_fit))
    mean = x.mean(axis=0, keepdims=True)
    scale = x.std(axis=0, keepdims=True)
    scale = np.where(scale < 1e-6, 1.0, scale)
    x = ((x - mean) / scale).astype(np.float32, copy=False)
    n_components = int(min(n_components, x.shape[0] - 1, x.shape[1]))
    if n_components < 2:
        raise ValueError('Need at least 3 real samples to fit PCA.')

    try:
        from sklearn.decomposition import PCA
        pca = PCA(n_components=n_components, svd_solver='randomized', random_state=0)
        pca.fit(x)
        components = pca.components_
        evr = pca.explained_variance_ratio_
    except Exception as exc:
        print('sklearn PCA unavailable or failed; using numpy SVD:', repr(exc))
        _, s, vt = np.linalg.svd(x, full_matrices=False)
        components = vt[:n_components]
        var = (s ** 2) / max(len(x) - 1, 1)
        evr = var[:n_components] / np.clip(var.sum(), 1e-30, None)
    return PCAEncoder(mean.squeeze(0), scale.squeeze(0), components, evr)


def l2_normalize(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(norm, 1e-12, None)


def nearest_real_neighbor_similarity(real_z: np.ndarray, batch_size: int = 1024) -> np.ndarray:
    real_z = np.asarray(real_z, dtype=np.float32)
    out = []
    n = len(real_z)
    for start in range(0, n, batch_size):
        stop = min(start + batch_size, n)
        sim = real_z[start:stop] @ real_z.T
        rows = np.arange(stop - start)
        cols = np.arange(start, stop)
        sim[rows, cols] = -np.inf
        out.append(sim.max(axis=1))
    return np.concatenate(out)


def generated_nearest_similarity(gen_z: np.ndarray, real_z: np.ndarray, batch_size: int = 1024) -> np.ndarray:
    out = []
    for start in range(0, len(gen_z), batch_size):
        stop = min(start + batch_size, len(gen_z))
        sim = gen_z[start:stop] @ real_z.T
        out.append(sim.max(axis=1))
    return np.concatenate(out)


def pca_generalization_metrics(real_emb: np.ndarray, gen_emb: np.ndarray, quantile: float = 0.99) -> dict[str, float]:
    real_z = l2_normalize(real_emb)
    gen_z = l2_normalize(gen_emb)
    real_nn = nearest_real_neighbor_similarity(real_z, batch_size=SIMILARITY_BATCH_SIZE)
    finite_real_nn = real_nn[np.isfinite(real_nn)]
    if len(finite_real_nn) == 0:
        raise ValueError('Need at least two real samples for real-real threshold calibration.')
    threshold = float(np.quantile(finite_real_nn, quantile))
    max_sim = generated_nearest_similarity(gen_z, real_z, batch_size=SIMILARITY_BATCH_SIZE)
    copy_fraction = float(np.mean(max_sim >= threshold))
    return {
        'pca_copy_threshold': threshold,
        'pca_copy_fraction': copy_fraction,
        'pca_generalization_score': 1.0 - copy_fraction,
        'pca_real_nn_median': float(np.median(finite_real_nn)),
        'pca_real_nn_q95': float(np.quantile(finite_real_nn, 0.95)),
        'pca_max_sim_median': float(np.median(max_sim)),
        'pca_max_sim_q90': float(np.quantile(max_sim, 0.90)),
        'pca_max_sim_q95': float(np.quantile(max_sim, 0.95)),
        'pca_max_sim_q99': float(np.quantile(max_sim, 0.99)),
    }

## Load Real Data And Fit PCA

For `PCA_FIT_MODE="largest"`, this cell fits one PCA encoder for each architecture using the largest completed run for that architecture. If you set `PCA_FIT_MODE="per_run"`, the metric cell will fit a separate encoder for every run, which is useful for debugging but less clean for a dataset-size curve.

In [ ]:
real_cache: dict[tuple[str, int | None], np.ndarray] = {}
encoder_cache: dict[str, dict[str, Any]] = {}


def load_real_for_row(row: dict[str, Any], *, slice_cap: int | None) -> np.ndarray:
    key = (row['run_name'], slice_cap)
    if key in real_cache:
        return real_cache[key]
    config_path = config_path_for(row)
    raw_cap = raw_sim_cap_for(row, slice_cap)
    real = load_real_from_config(config_path, max_raw_samples=raw_cap)
    real = evenly_limit(as_nchw(real), slice_cap)
    real_cache[key] = real
    return real


if PCA_FIT_MODE not in {'largest', 'per_run'}:
    raise ValueError('PCA_FIT_MODE must be "largest" or "per_run".')

if PCA_FIT_MODE == 'largest':
    for arch in ARCHES:
        arch_rows = [row for row in rows if row.get('arch') == arch and sample_path_for(row) is not None and config_path_for(row).exists()]
        if not arch_rows:
            print('No completed rows for PCA fit:', arch)
            continue
        fit_row = max(arch_rows, key=dataset_size)
        print(f"Fitting {arch} PCA on {fit_row['run_name']} with up to {PCA_MAX_FIT_REAL} real slices")
        fit_real = load_real_for_row(fit_row, slice_cap=PCA_MAX_FIT_REAL)
        encoder = fit_pca_encoder(fit_real, n_components=PCA_N_COMPONENTS, max_fit=PCA_MAX_FIT_REAL)
        encoder_cache[arch] = {
            'encoder': encoder,
            'fit_run_name': fit_row['run_name'],
            'fit_dataset_size': dataset_size(fit_row),
            'n_fit_real': len(fit_real),
        }
        print(
            f"  components={len(encoder.explained_variance_ratio)} "
            f"explained_var_sum={encoder.explained_variance_ratio.sum():.3f}"
        )

## Compute PCA Generalizability Curve

This writes the main PCA table. The score is `pca_generalization_score = 1 - pca_copy_fraction`; higher means fewer generated samples are copy-like under the PCA nearest-neighbor threshold.

In [ ]:
records = []

for row in rows:
    run_name = row['run_name']
    arch = row.get('arch')
    sample_path = sample_path_for(row)
    config_path = config_path_for(row)
    if sample_path is None or not config_path.exists():
        print('SKIP missing inputs:', run_name, 'sample=', sample_path, 'config exists=', config_path.exists())
        continue

    generated = as_nchw(load_sample_array(sample_path))
    generated = evenly_limit(generated, PCA_MAX_GENERATED)
    real_training = load_real_for_row(row, slice_cap=PCA_MAX_REAL_COMPARE)

    if PCA_FIT_MODE == 'largest':
        if arch not in encoder_cache:
            print('SKIP no PCA encoder for arch:', arch)
            continue
        encoder_info = encoder_cache[arch]
        encoder = encoder_info['encoder']
    else:
        print(f'Fitting per-run PCA on {run_name}')
        fit_real = load_real_for_row(row, slice_cap=PCA_MAX_FIT_REAL)
        encoder = fit_pca_encoder(fit_real, n_components=PCA_N_COMPONENTS, max_fit=PCA_MAX_FIT_REAL)
        encoder_info = {
            'fit_run_name': run_name,
            'fit_dataset_size': dataset_size(row),
            'n_fit_real': len(fit_real),
        }

    print(
        f"{run_name}: arch={arch} generated={len(generated)} "
        f"real_training={len(real_training)} dataset_size={dataset_size(row):.0f}"
    )
    real_emb = encoder.transform(real_training)
    gen_emb = encoder.transform(generated)
    metrics = pca_generalization_metrics(real_emb, gen_emb, quantile=PCA_COPY_QUANTILE)

    record = {
        'run_name': run_name,
        'arch': arch,
        'arch_label': ARCH_LABELS.get(arch, arch),
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'n_generated': len(generated),
        'n_real_training_compared': len(real_training),
        'pca_components': len(encoder.explained_variance_ratio),
        'pca_explained_variance_sum': float(encoder.explained_variance_ratio.sum()),
        'pca_fit_mode': PCA_FIT_MODE,
        'pca_fit_run_name': encoder_info['fit_run_name'],
        'pca_fit_dataset_size': encoder_info['fit_dataset_size'],
        'n_pca_fit_real': encoder_info['n_fit_real'],
        'pca_copy_quantile': PCA_COPY_QUANTILE,
        'real_slice_cap': PCA_MAX_REAL_COMPARE,
        'generated_slice_cap': PCA_MAX_GENERATED,
        'sample_path': str(sample_path),
        **metrics,
    }
    records.append(record)
    print(
        f"  PCA_GL={record['pca_generalization_score']:.3f} "
        f"copy_fraction={record['pca_copy_fraction']:.3f} "
        f"threshold={record['pca_copy_threshold']:.3f} "
        f"max_sim_median={record['pca_max_sim_median']:.3f}"
    )

if not records:
    raise RuntimeError('No PCA records were computed. Check the status table for missing sample/config files.')

df = pd.DataFrame(records).sort_values(['arch', 'dataset_size']).reset_index(drop=True)
display(df)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
out_csv = TABLE_DIR / 'figure2b_style_generalizability_pca.csv'
df.to_csv(out_csv, index=False)
print('saved', out_csv)

## Figure 2(b)-Style PCA Plot

This is the PCA version of the generalizability curve. Use it beside the SSCD figure, not as a direct replacement for the paper metric.

In [ ]:
def xfmt(x: float, _pos: int) -> str:
    if x <= 0:
        return ''
    exponent = int(round(np.log2(x)))
    if np.isclose(x, 2**exponent):
        return rf'$2^{{{exponent}}}$'
    return f'{x:g}'

fig, ax = plt.subplots(figsize=(7.4, 5.1))
plot_df = df.sort_values(['arch', 'dataset_size'])

for arch in ARCHES:
    sub = plot_df[plot_df['arch'] == arch].sort_values('dataset_size')
    if sub.empty:
        continue
    ax.plot(
        sub['dataset_size'],
        sub['pca_generalization_score'],
        color=ARCH_COLORS.get(arch),
        marker=ARCH_MARKERS.get(arch, 'o'),
        lw=2.5,
        ms=7,
        label=ARCH_LABELS.get(arch, arch),
    )

ax.set_xscale('log', base=2)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('dataset size')
ax.set_ylabel('PCA generalizability')
ax.set_title(f'Figure 2(b)-style PCA GL, real-NN q={PCA_COPY_QUANTILE:g}')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(xfmt))
ax.grid(alpha=0.25)
ax.legend(title='model')

note = f'PCA fit: {PCA_FIT_MODE}; real comparison cap: {PCA_MAX_REAL_COMPARE} slices'
ax.text(0.02, 0.03, note, transform=ax.transAxes, fontsize=9, color='dimgray')

fig.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_png = OUTPUT_DIR / 'figure2b_style_generalizability_pca.png'
out_pdf = OUTPUT_DIR / 'figure2b_style_generalizability_pca.pdf'
fig.savefig(out_png, dpi=180, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
print('saved', out_png)
print('saved', out_pdf)
plt.show()

## Diagnostics: Copy Fraction And PCA Similarity

These explain the PCA GL score. `copy_fraction` moves opposite to the main score. The right panel shows generated-to-training nearest-neighbor similarity plus the run-specific real-real threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.3), sharex=True)
plot_df = df.sort_values(['arch', 'dataset_size'])

for arch in ARCHES:
    sub = plot_df[plot_df['arch'] == arch].sort_values('dataset_size')
    if sub.empty:
        continue
    label = ARCH_LABELS.get(arch, arch)
    color = ARCH_COLORS.get(arch)
    marker = ARCH_MARKERS.get(arch, 'o')
    axes[0].plot(sub['dataset_size'], sub['pca_copy_fraction'], marker=marker, lw=2, color=color, label=label)
    axes[1].plot(sub['dataset_size'], sub['pca_max_sim_median'], marker=marker, lw=2, color=color, label=f'{label} generated median')
    axes[1].plot(sub['dataset_size'], sub['pca_copy_threshold'], marker=marker, lw=1.5, ls=':', color=color, alpha=0.8, label=f'{label} threshold')

axes[0].set_xscale('log', base=2)
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_xlabel('dataset size')
axes[0].set_ylabel('PCA copy fraction')
axes[0].set_title(f'fraction above real-NN q={PCA_COPY_QUANTILE:g}')
axes[0].xaxis.set_major_formatter(ticker.FuncFormatter(xfmt))
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].set_xscale('log', base=2)
axes[1].set_xlabel('dataset size')
axes[1].set_ylabel('PCA nearest-neighbor cosine')
axes[1].set_title('nearest-training similarity and threshold')
axes[1].xaxis.set_major_formatter(ticker.FuncFormatter(xfmt))
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)

fig.tight_layout()
out_diag = OUTPUT_DIR / 'figure2b_style_generalizability_pca_diagnostics.png'
fig.savefig(out_diag, dpi=180, bbox_inches='tight')
print('saved', out_diag)
plt.show()

## Optional: PCA Embedding Scatter

This quick visualization shows the first two PCA coordinates for one small-data and one large-data run per architecture. It is only a qualitative check; the metric above uses all PCA components.

In [ ]:
for arch in ARCHES:
    sub_rows = [row for row in rows if row.get('arch') == arch and sample_path_for(row) is not None and config_path_for(row).exists()]
    if not sub_rows:
        continue
    chosen = []
    chosen.append(min(sub_rows, key=dataset_size))
    largest = max(sub_rows, key=dataset_size)
    if largest['run_name'] != chosen[0]['run_name']:
        chosen.append(largest)

    fig, axes = plt.subplots(1, len(chosen), figsize=(5.2 * len(chosen), 4.4), squeeze=False)
    for ax, row in zip(axes.ravel(), chosen):
        this_arch = row.get('arch')
        if PCA_FIT_MODE == 'largest':
            encoder = encoder_cache[this_arch]['encoder']
        else:
            encoder = fit_pca_encoder(load_real_for_row(row, slice_cap=PCA_MAX_FIT_REAL), PCA_N_COMPONENTS, PCA_MAX_FIT_REAL)
        real = load_real_for_row(row, slice_cap=min(PCA_MAX_REAL_COMPARE, 1024))
        generated = evenly_limit(as_nchw(load_sample_array(sample_path_for(row))), PCA_MAX_GENERATED)
        real_emb = encoder.transform(real)
        gen_emb = encoder.transform(generated)
        ax.scatter(real_emb[:, 0], real_emb[:, 1], s=8, alpha=0.25, color='black', label='real training')
        ax.scatter(gen_emb[:, 0], gen_emb[:, 1], s=20, alpha=0.75, color='tab:blue', label='generated')
        ax.set_title(f"{row['run_name']}\nN={dataset_size(row):.0f}")
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)
    fig.suptitle(f'{ARCH_LABELS.get(arch, arch)} PCA embedding scatter')
    fig.tight_layout()
    out_scatter = OUTPUT_DIR / f'figure2b_style_generalizability_pca_scatter_{arch}.png'
    fig.savefig(out_scatter, dpi=180, bbox_inches='tight')
    print('saved', out_scatter)
    plt.show()

## Notes For Interpreting This Plot

- SSCD uses a natural-image copy-detection encoder and a fixed paper threshold.
- PCA uses a CAMELS-field encoder fitted from the normalized training data and a threshold calibrated from real-real nearest-neighbor similarity.
- A high PCA GL means generated samples are not unusually close to the training set in the retained linear modes. It does not guarantee good one-point statistics or P(k).
- If the PCA and SSCD curves disagree, inspect images, one-point histograms, P(k), and nearest-neighbor examples before treating either score as definitive.